# MD-GRTN — Exact Paper Dataset Construction

**3 sequences exactly as paper Section III-A + Fig 4(a):**

| Sequence | Paper name | Window | Offset |
|---|---|---|---|
| `X_RecN` | Neighboring sequence | `t-12 → t` | 0 steps |
| `X_HourN` | Hourly sequence | `t-24 → t-12` | **12 steps = 1 hr ago** |
| `X_DayN` | Daily sequence | `t-300 → t-288` | **288 steps = 24 hrs ago** |

**Paper settings (Section V-B-1):**
- Gaussian noise μ=0, σ=10 added to inputs (labels stay clean)
- Z-score normalisation before input
- Split: 7:1:2
- AdamW lr=0.001, batch=64, heads=3 (d_model=96, 96÷3=32 ✓), L=3, MGRC=6
- Huber loss, 800 iterations

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} ✓')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# ══════════════════════════════════════════════════
# CONFIG — exact paper Section V-B-1
# ══════════════════════════════════════════════════
class Config:
    # data
    data_path   = "/content/PEMS08.npz"
    num_nodes   = 170
    in_features = 3               # flow, occupancy, speed
    seq_len     = 12              # Th = 12 steps (paper)
    pred_len    = 12              # Tf = 12 steps (paper)
    feature_idx = 0               # predict flow channel

    # ── 3-period offsets — EXACT paper Fig 4(a) ──
    # X_RecN  : neighboring sequence → t-12 to t        (offset = 0)
    # X_HourN : hourly sequence      → t-24 to t-12     (offset = 12 steps = 1 hr)
    # X_DayN  : daily sequence       → t-300 to t-288   (offset = 288 steps = 24 hr)
    offset_rec  = 0
    offset_hour = 12    # 12 × 5min = 1 hour ago
    offset_day  = 288   # 288 × 5min = 24 hours ago

    # noise — paper Section V-A: Gaussian μ=0, σ=10 for PEMS
    noise_mean  = 0.0
    noise_std   = 10.0

    # model — paper Section V-B-1
    # d_model=96 divisible by n_heads=3 → head_dim=32 ✓
    d_model     = 96
    n_heads     = 3     # paper: 3 heads
    n_seqs      = 3
    num_layers  = 3     # STFormer L=3 (paper)
    gru_layers  = 6     # MGRC 6 layers (paper)
    dropout     = 0.1

    # training — paper Section V-B-1
    batch_size  = 64    # paper: batch=64
    lr          = 1e-3  # paper: lr=0.001
    weight_decay= 0.01  # paper: wd=0.01
    max_iters   = 800   # paper: 800 iterations
    patience    = 15
    delta_h     = 1.0   # Huber δ
    train_ratio = 0.7   # paper: 7:1:2
    val_ratio   = 0.1

    # checkpoint
    ckpt_path = 'mdgrtn_paper_ckpt.pt'
    best_path = 'mdgrtn_paper_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert cfg.d_model % cfg.n_heads == 0, f"{cfg.d_model} not divisible by {cfg.n_heads}"
print(f'Config ready | d_model={cfg.d_model} heads={cfg.n_heads} head_dim={cfg.d_model//cfg.n_heads} ✓')
print(f'Offsets → rec={cfg.offset_rec} | hour={cfg.offset_hour} steps (1hr) | day={cfg.offset_day} steps (24hr)')
print(f'Noise   → μ={cfg.noise_mean} σ={cfg.noise_std} (paper Section V-A)')
print(f'Split   → {cfg.train_ratio*100:.0f}/{cfg.val_ratio*100:.0f}/{(1-cfg.train_ratio-cfg.val_ratio)*100:.0f} (paper 7:1:2)')
print(f'Device  → {device}')

In [ ]:
# ══════════════════════════════════════════════════
# DATA — exact paper Section V-A
# Z-score normalisation → add noise to INPUTS only
# Labels (y) stay CLEAN (noise-free)
# ══════════════════════════════════════════════════

def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    print('Keys:', list(raw.keys()))
    data = raw['data'].astype(np.float32)   # (T, N, F)
    T, N, F = data.shape
    print(f'Raw shape: {data.shape}')

    # Z-score normalisation (paper Section V-A)
    mean = data.mean(axis=0)                      # (N, F) per-sensor
    std  = data.std(axis=0)  + 1e-8               # (N, F)
    data_norm = (data - mean[None]) / std[None]   # (T, N, F) clean normalised

    # Adjacency matrix
    for key in ('adjacency_matrix','adj_mx','adj'):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f'Adjacency from "{key}" shape={A_dist.shape}')
            break
    else:
        print('No adjacency found — using identity')
        A_dist = np.eye(N, dtype=np.float32)

    deg    = A_dist.sum(axis=1, keepdims=True) + 1e-8
    A_dist = A_dist / deg
    return data_norm, mean, std, A_dist


class PaperDataset(Dataset):
    """
    Exact paper dataset construction (Section III-A + Fig 4a).

    For each sample at time t:
      X_RecN  = data[t-12    : t]        + noise   (recent, neighboring)
      X_HourN = data[t-24    : t-12]     + noise   (1 hour ago, hourly)
      X_DayN  = data[t-300   : t-288]    + noise   (24 hours ago, daily)
      y       = data[t       : t+12, fi]           (clean, noise-free label)

    Noise: Gaussian μ=0, σ=10 added to inputs ONLY (paper Section V-A)
    """
    def __init__(self, data, cfg, start, end):
        self.data  = torch.from_numpy(data)
        self.S     = cfg.seq_len           # 12
        self.P     = cfg.pred_len          # 12
        self.fi    = cfg.feature_idx       # 0
        self.oh    = cfg.offset_hour       # 12
        self.od    = cfg.offset_day        # 288
        self.nm    = cfg.noise_mean        # 0.0
        self.ns    = cfg.noise_std         # 10.0

        # Earliest valid t:
        # Need day window: t - S - od >= 0  →  t >= S + od = 12 + 288 = 300
        min_t = cfg.seq_len + cfg.offset_day
        self.idx = list(range(start + min_t, end - cfg.pred_len + 1))

    def __len__(self): return len(self.idx)

    def _add_noise(self, x):
        """Gaussian noise μ=0, σ=10 — paper Section V-A."""
        return x + torch.randn_like(x) * self.ns + self.nm

    def __getitem__(self, i):
        t = self.idx[i]
        S, P, fi = self.S, self.P, self.fi

        # Clean windows
        x_rec_clean  = self.data[t - S          : t]               # (S,N,F)
        x_hour_clean = self.data[t - S - self.oh : t - self.oh]    # (S,N,F) 1hr ago
        x_day_clean  = self.data[t - S - self.od : t - self.od]    # (S,N,F) 24hr ago
        y            = self.data[t               : t + P, :, fi]   # (P,N) clean

        # Add noise to inputs (paper: noisy inputs, clean labels)
        x_rec  = self._add_noise(x_rec_clean)
        x_hour = self._add_noise(x_hour_clean)
        x_day  = self._add_noise(x_day_clean)

        return x_rec, x_hour, x_day, y


def build_dataloaders(cfg):
    set_seed()
    data, mean, std, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))

    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    dl_tr = DataLoader(PaperDataset(data,cfg,0, t1), shuffle=True,  **kw)
    dl_va = DataLoader(PaperDataset(data,cfg,t1,t2), shuffle=False, **kw)
    dl_te = DataLoader(PaperDataset(data,cfg,t2,T),  shuffle=False, **kw)

    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    print(f'Min valid t = {cfg.seq_len + cfg.offset_day} (need 24hr history)')
    return dl_tr, dl_va, dl_te, mean, std, A_dist

print('Paper dataset ready.')

In [ ]:
# ══════════════════════════════════════════════════
# MDAF — 3 BackNets + MAF  (paper Section IV-A)
# BackNet_k denoises X_k → X̂_k  (Eq. 4)
# MAF fuses X̂_Rec, X̂_Hour, X̂_Day → X_F1 (Eq. 5-9)
# ══════════════════════════════════════════════════

class BackNet(nn.Module):
    """MLP denoiser — paper Eq.4: X̂_k = BackNet_k(X_k)"""
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim),
        )
    def forward(self, x): return self.net(x)   # (B,S,N,F)


class MAFModule(nn.Module):
    """
    Per-period Q,K,V → scaled dot-product → Concat → FC
    Paper Eq. 5-9, architecture figure: 3 Linear → DotProduct → Concat+FC
    """
    def __init__(self, in_dim, d_model, n_heads, n_seqs=3):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_j    = d_model // n_heads
        # Per-period separate W_Q, W_K, W_V (paper Eq. 5-7)
        self.W_Q = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_K = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_V = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        # Concat + FC = W_MH (paper Eq. 9)
        self.W_MH = nn.Linear(d_model * n_seqs, d_model)

    def forward(self, seqs):
        """seqs: list of 3 × (B,S,N,F) → X_F1: (B,S,N,d)"""
        B, S, N, _ = seqs[0].shape
        heads = []
        for i, x in enumerate(seqs):
            f = x.reshape(B*S, N, -1)
            Q = self.W_Q[i](f)
            K = self.W_K[i](f)
            V = self.W_V[i](f)
            # Scaled dot-product (paper Eq. 8)
            sc = torch.bmm(Q, K.transpose(1,2)) / (self.d_j ** 0.5)
            h  = torch.bmm(F.softmax(sc, dim=-1), V)   # (B*S, N, d)
            heads.append(h.reshape(B, S, N, -1))
        # Concat + W_MH (paper Eq. 9)
        return self.W_MH(torch.cat(heads, dim=-1))      # (B,S,N,d)


class MDModule(nn.Module):
    """3 separate BackNets — BackNet_1, BackNet_2, BackNet_3"""
    def __init__(self, in_features, d_model, n_seqs=3):
        super().__init__()
        self.backnets = nn.ModuleList([BackNet(in_features, d_model) for _ in range(n_seqs)])
    def forward(self, seqs):
        return [bn(s) for bn, s in zip(self.backnets, seqs)]


class MDAFModule(nn.Module):
    def __init__(self, in_features, d_model, n_heads, n_seqs=3):
        super().__init__()
        self.md  = MDModule(in_features, d_model, n_seqs)
        self.maf = MAFModule(in_features, d_model, n_heads, n_seqs)
    def forward(self, seqs):
        cleaned = self.md(seqs)       # denoise each period
        return self.maf(cleaned)      # fuse → X_F1

print('MDAF defined (3 BackNets + MAF, paper Eq.4-9).')

In [ ]:
# ══════════════════════════════════════════════════
# MGRC — paper Eq.10-14, 6 stacked layers
# A_dyna = softmax(ReLU(E1@E2^T))  E1,E2 ∈ R^{N×1}
# A_F    = ReLU(Conv2D(Concat(A_dyna, A_dist)))
# X'     = ReLU(A_F @ X @ W_GCN)
# GRU update
# ══════════════════════════════════════════════════

class MultiGraphFusion(nn.Module):
    def __init__(self, N):
        super().__init__()
        self.E1     = nn.Parameter(torch.randn(N, 1) * 0.01)
        self.E2     = nn.Parameter(torch.randn(N, 1) * 0.01)
        self.conv2d = nn.Conv2d(2, 1, kernel_size=1)

    def forward(self, A_dist):
        # Eq.10: A_dyna = softmax(ReLU(E1 @ E2^T))
        A_dyna  = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        # Eq.12: A_F = ReLU(Conv2D(Concat(A_dyna, A_dist)))
        stack   = torch.stack([A_dyna, A_dist], dim=0).unsqueeze(0)
        A_F     = F.relu(self.conv2d(stack).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)


class GCN_GRU_Layer(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.W_GCN = nn.Linear(in_dim, hid_dim, bias=False)
        self.gru   = nn.GRUCell(hid_dim, hid_dim)

    def forward(self, x_seq, A_F):
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B*N, self.gru.hidden_size, device=x_seq.device)
        outs = []
        for t in range(S):
            # Eq.13: X' = ReLU(A_F @ X @ W_GCN)
            agg = torch.einsum('nm,bmd->bnd', A_F, x_seq[:,t])
            xp  = F.relu(self.W_GCN(agg))
            # Eq.14: GRU
            h   = self.gru(xp.reshape(B*N,-1), h)
            outs.append(h.reshape(B,N,-1))
        return torch.stack(outs, dim=1)


class MGRCModule(nn.Module):
    def __init__(self, in_dim, hid_dim, N, n_layers=6):
        super().__init__()
        self.fusion = MultiGraphFusion(N)
        dims = [in_dim] + [hid_dim]*n_layers
        self.layers = nn.ModuleList([
            GCN_GRU_Layer(dims[i], dims[i+1]) for i in range(n_layers)])

    def forward(self, X_F1, A_dist):
        A_F = self.fusion(A_dist)
        x   = X_F1
        for layer in self.layers:
            x = layer(x, A_F)
        return x

print('MGRC defined (6 layers, dual adjacency, paper Eq.10-14).')

In [ ]:
# ══════════════════════════════════════════════════
# STFormer — paper Eq.15-26, L=3 layers
# ══════════════════════════════════════════════════

class SpatialTransformerLayer(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d, d*4), nn.ReLU(), nn.Linear(d*4, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dr)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.reshape(B*S, N, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, S, N, d)


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d, d*4), nn.ReLU(), nn.Linear(d*4, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dr)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.permute(0,2,1,3).reshape(B*N, S, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print('STFormer layers defined.')

In [ ]:
class MDGRTN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        L, h, dr = cfg.num_layers, cfg.n_heads, cfg.dropout
        P, S = cfg.pred_len, cfg.seq_len

        self.mdaf = MDAFModule(F, d, h, n_seqs=3)
        self.mgrc = MGRCModule(d, d, N, n_layers=cfg.gru_layers)

        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, N, d) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, S, 1, d) * 0.02)

        self.spatial_layers  = nn.ModuleList([SpatialTransformerLayer(d, h, dr) for _ in range(L)])
        self.temporal_layers = nn.ModuleList([TemporalTransformerLayer(d, h, dr) for _ in range(L)])

        self.out_proj = nn.Linear(d * S, P)

    def forward(self, x_rec, x_hour, x_day, A_dist):
        """
        x_rec, x_hour, x_day : (B, S, N, F) — noisy inputs
        A_dist               : (N, N)
        returns              : (B, P, N)
        """
        X_F1 = self.mdaf([x_rec, x_hour, x_day])   # (B,S,N,d)
        X_F2 = self.mgrc(X_F1, A_dist)              # (B,S,N,d)

        x = X_F2 + self.spatial_pos
        for layer in self.spatial_layers:
            x = layer(x)
        x = x + self.temporal_pos
        for layer in self.temporal_layers:
            x = layer(x)

        B, S, N, d = x.shape
        return self.out_proj(x.permute(0,2,1,3).reshape(B,N,S*d)).permute(0,2,1)


set_seed()
model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')
print(f'  n_seqs=3 | num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers} | d_model={cfg.d_model}')

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, dummy, dummy, adj)
print(f'Output: {out.shape}  ✓  (expected [2, {cfg.pred_len}, {cfg.num_nodes}])')

In [ ]:
def masked_mae(pred, true):
    return torch.abs(pred - true).mean()

def masked_rmse(pred, true):
    return ((pred - true)**2).mean().sqrt()

def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

def huber_loss(pred, true, delta=1.0):
    return F.huber_loss(pred, true, delta=delta)

print('Metrics defined.')

In [ ]:
# ── Mount Drive (uncomment) ──
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

# Per-sensor denorm — (N,) for flow channel
mean_flow = torch.from_numpy(mean_np[:,cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:,cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)

print(f'mean_flow: min={mean_flow.min():.2f}  max={mean_flow.max():.2f}')
print(f'std_flow:  min={std_flow.min():.2f}   max={std_flow.max():.2f}')
print(f'Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}')

In [ ]:
def train_epoch(model, loader, optimizer, A_dist, device, delta_h):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_day  = x_day.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, x_day, A_dist)
        loss   = huber_loss(pred, y, delta_h)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, x_day, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_day  = x_day.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, x_day, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train/eval defined.')

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_mae, history, cfg, drive_path=None):
    torch.save({
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch': epoch, 'best_mae': best_mae,
        'history': history, 'seed': SEED,
    }, cfg.ckpt_path)
    if drive_path:
        import shutil; shutil.copy(cfg.ckpt_path, drive_path)
        print(f'Saved to Drive ✓', end='\r')
    else:
        print(f'Ckpt saved ep={epoch} mae={best_mae:.3f}', end='\r')


def load_checkpoint(model, optimizer, scheduler, cfg, drive_path=None):
    if drive_path:
        import shutil; shutil.copy(drive_path, cfg.ckpt_path)
    if not os.path.exists(cfg.ckpt_path):
        print('No checkpoint — starting fresh.')
        return 1, float('inf'), {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}
    ckpt = torch.load(cfg.ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    print(f'Resumed epoch {ckpt["epoch"]} | best_mae={ckpt["best_mae"]:.3f}')
    return ckpt['epoch']+1, ckpt['best_mae'], ckpt['history']

print('Checkpoint utilities ready.')

In [ ]:
set_seed()

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-6)

# ── To RESUME uncomment: ──
# start_epoch, best_val_mae, history = load_checkpoint(
#     model, optimizer, scheduler, cfg,
#     drive_path='/content/drive/MyDrive/mdgrtn_paper_ckpt.pt')

DRIVE_CKPT   = None   # set Drive path to auto-save
start_epoch  = 1
best_val_mae = float('inf')
patience_cnt = 0
history      = {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}

# Iter-based: paper says 800 iterations = iters not epochs
iters_per_ep = len(dl_train)
max_epochs   = max(1, cfg.max_iters // iters_per_ep)
# Also run up to 100 epochs with early stopping
max_epochs   = max(max_epochs, 100)

print(f'Paper training | iters={cfg.max_iters} → ~{iters_per_ep} iters/ep → ~{cfg.max_iters//iters_per_ep} epochs')
print(f'With patience={cfg.patience} → max {max_epochs} epochs')
print(f'Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*68)

total_iters = 0
for epoch in range(start_epoch, max_epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device, cfg.delta_h)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step()
    total_iters += iters_per_ep

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  ← best ✓'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    print(f'Ep {epoch:03d} (~{total_iters} iters) | Loss={train_loss:.4f} | '
          f'MAE={val_mae:.3f}  RMSE={val_rmse:.3f}  MAPE={val_mape:.2f}%{tag}')

    save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_mae, history, cfg, drive_path=DRIVE_CKPT)

print(f'\nBest Val MAE: {best_val_mae:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', lw=2, label='Huber Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', lw=2, label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', lw=1.5, label='Paper 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', lw=2, label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', lw=1.5, label='Paper 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_paper.png', dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_day  = x_day.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, x_day, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*60)
    print('  TEST RESULTS  —  paper-style (all 12 steps averaged)')
    print('='*60)
    print(f'  MAE  : {mae:.3f}    paper: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    paper: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   paper:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*60)
    return mae, rmse, mape

paper_style_eval(model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, x_day, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_day  = x_day.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, x_day, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)